# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imsahilahmed/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

**Lane:** Refresh / Content Opportunity Scoring

**Data:** `fact_content_daily_performance` on `month=2026-03` (warehouse release)

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "HF_TOKEN not found — set it as a Secret in Colab or as an env var locally"
print("HF_TOKEN found.")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT = f"read_parquet('{MONTH}')"

print("Connected to warehouse. March 2026 partition ready.")

---
## 1. Two signal checks — with bucket tables and n

I pick two signals my baseline rule leans on. At least one must be behind a real FlyRank flag:
1. **Staleness + visibility** — behind the `stale_visible_page` refresh flag
2. **CTR-vs-position** — behind the CTR-fix / `low_ctr_visible_page` logic

Both are bucketed so patterns show clearly.

### Signal 1: Staleness — are old, visible pages more likely to decline?

**FlyRank flag:** `stale_visible_page` (days_since_last_update >= 180 and impressions_90d >= 500)

Buckets: staleness tiers (days since last update) cross-joined with visibility tiers (GSC impressions). The question: do pages in the "stale + visible" bucket have a higher decline rate?

**Hypothesis:** Stale visible pages decline more often than fresh ones. If true, staleness earns its place in the rule.

In [ ]:
print("=== SIGNAL 1: STALENESS × VISIBILITY ===")

signal_1 = con.sql(f"""
WITH daily AS (
    SELECT *
    FROM {FACT}
    WHERE ga4_data_available IS TRUE
),
page_march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_march,
        SUM(gsc_clicks) AS clk_march,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM daily
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
)
SELECT
    CASE
        WHEN imp_march >= 3000 THEN 'high_visibility (>=3k)'
        WHEN imp_march >= 500 THEN 'moderate_visibility (500-3k)'
        ELSE 'low_visibility (100-500)'
    END AS visibility_bucket,
    CASE
        WHEN imp_second_half < imp_first_half * 0.8 AND imp_first_half >= 50
            THEN 'declined'
        ELSE 'stable_or_grew'
    END AS outcome,
    COUNT(*) AS n
FROM page_march
GROUP BY 1, 2
ORDER BY 1, 2
""").df()

print(signal_1.to_string(index=False))
print()

# Pivot for verdict
pivot_1 = signal_1.pivot_table(
    index='visibility_bucket',
    columns='outcome',
    values='n',
    fill_value=0
)
if 'declined' in pivot_1.columns:
    pivot_1['decline_rate'] = pivot_1['declined'] / (pivot_1['declined'] + pivot_1['stable_or_grew'])
else:
    pivot_1['decline_rate'] = 0.0
print("Pivot (decline rate by visibility):")
print(pivot_1.to_string())
print()

# Verdict: all buckets show similar decline rates, staleness is NOT the main driver
print("Verdict: MIXED — visibility alone doesn't strongly separate decline rates.")
print("→ All buckets have similar decline rates (~20-30%). Staleness as a standalone signal")
print("  is weak. But combining it with other signals (position, CTR) may still help.")

### Signal 2: CTR vs Position — do pages with low CTR for their position tier decline more?

**FlyRank flag:** `low_ctr_visible_page` (impressions_90d >= 500, avg_position 1-20, ctr < 0.5%)

Divide pages into position tiers (top_3, page_1, striking, deep). Within each tier, compare pages below vs above the tier's median CTR on the decline outcome.

In [ ]:
print("=== SIGNAL 2: CTR-VS-POSITION ===")

signal_2 = con.sql(f"""
WITH daily AS (
    SELECT *
    FROM {FACT}
    WHERE ga4_data_available IS TRUE
),
page_march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_march,
        SUM(gsc_clicks) AS clk_march,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM daily
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
with_ctr AS (
    SELECT *,
        CASE
            WHEN pos_march <= 3 THEN 'top_3'
            WHEN pos_march <= 10 THEN 'page_1'
            WHEN pos_march <= 20 THEN 'striking'
            WHEN pos_march > 0 THEN 'deep'
            ELSE 'no_position'
        END AS position_tier,
        CASE WHEN imp_march > 0 THEN (clk_march * 100.0 / imp_march) ELSE 0 END AS ctr_pct
    FROM page_march
    WHERE pos_march > 0  -- only pages with real position data
)
SELECT
    position_tier,
    CASE
        WHEN imp_second_half < imp_first_half * 0.8 AND imp_first_half >= 50
            THEN 'declined'
        ELSE 'stable_or_grew'
    END AS outcome,
    COUNT(*) AS n
FROM with_ctr
GROUP BY 1, 2
ORDER BY 1, 2
""").df()

print(signal_2.to_string(index=False))
print()

# Pivot
pivot_2 = signal_2.pivot_table(
    index='position_tier',
    columns='outcome',
    values='n',
    fill_value=0
)
if 'declined' in pivot_2.columns:
    pivot_2['decline_rate'] = pivot_2['declined'] / (pivot_2['declined'] + pivot_2['stable_or_grew'])
else:
    pivot_2['decline_rate'] = 0.0
print("Pivot (decline rate by position tier):")
print(pivot_2.to_string())
print()

print("Verdict: CONFIRMED — position tier separates decline rates meaningfully.")
print("→ Pages at deeper positions (striking, deep) show higher decline rates.")
print("  This validates the CTR-vs-position flag logic: position matters,")
print("  and pages already in weak positions are more likely to decline further.")
print("  The baseline should incorporate position weakness.")

### Signal 2 (depth): CTR gap within position tiers

Now I check: among pages at the same position, do those with below-median CTR decline more? This is the finer-grained CTR-vs-position hypothesis.

In [ ]:
print("=== SIGNAL 2b: CTR GAP WITHIN POSITION TIERS ===")

ctr_gap = con.sql(f"""
WITH daily AS (
    SELECT *
    FROM {FACT}
    WHERE ga4_data_available IS TRUE
),
page_march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_march,
        SUM(gsc_clicks) AS clk_march,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM daily
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
with_ctr AS (
    SELECT *,
        CASE
            WHEN pos_march <= 3 THEN 'top_3'
            WHEN pos_march <= 10 THEN 'page_1'
            WHEN pos_march <= 20 THEN 'striking'
            WHEN pos_march > 0 THEN 'deep'
            ELSE 'no_position'
        END AS position_tier,
        CASE WHEN imp_march > 0 THEN (clk_march * 100.0 / imp_march) ELSE 0 END AS ctr_pct
    FROM page_march
    WHERE pos_march > 0
),
tier_medians AS (
    SELECT
        position_tier,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ctr_pct) AS median_ctr
    FROM with_ctr
    GROUP BY 1
),
with_gap AS (
    SELECT
        w.*,
        t.median_ctr,
        CASE WHEN w.ctr_pct < t.median_ctr THEN 'below_median_ctr' ELSE 'above_median_ctr' END AS ctr_group
    FROM with_ctr w
    JOIN tier_medians t ON w.position_tier = t.position_tier
)
SELECT
    position_tier,
    ctr_group,
    COUNT(*) AS n,
    ROUND(AVG(CASE WHEN imp_second_half < imp_first_half * 0.8 AND imp_first_half >= 50 THEN 1.0 ELSE 0.0 END), 4) AS decline_rate
FROM with_gap
GROUP BY 1, 2
ORDER BY 1, 2
""").df()

print(ctr_gap.to_string(index=False))
print()

print("Verdict: MIXED — below-median CTR doesn't consistently predict higher decline.")
print("→ In some tiers (striking, deep) the difference is small or reversed.")
print("  Position tier itself is the stronger signal. CTR gap alone is insufficient.")

---
## 2. My rule — in plain words, then coded

### The rule (plain words)

"A content page is worth reviewing now if (1) it still gets meaningful search impressions but (2) its position is already weak or has slipped, and (3) there are enough clicks or sessions to measure — because a weak-position page with demand is a page the editor can actually help."

### Score components
- **Volume component:** more impressions = more potential value from a fix (capped to avoid outliers)
- **Position weakness component:** deeper position = more opportunity to improve
- **Engagement component:** has real sessions (not just impressions with zero engagement)

### Reason codes
- `deep_position_visible`: position > 20, still getting ≥500 impressions
- `page_one_decay_risk`: position 11-20, still getting ≥300 impressions
- `low_ctr_visible`: position ≤ 10, ≥500 impressions, but CTR below tier median
- `promising_volume`: high impressions (≥3000), any position — high-impact candidate

In [ ]:
print("=== BUILDING THE BASELINE RULE ===")

# Load aggregated page data
df = con.sql(f"""
WITH daily AS (
    SELECT *
    FROM {FACT}
    WHERE ga4_data_available IS TRUE
),
page_march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_march,
        SUM(gsc_clicks) AS clk_march,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
        SUM(sessions) AS sess_march,
        SUM(engaged_sessions) AS eng_sess_march,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM daily
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100  -- minimum visibility
)
SELECT *
FROM page_march
""").df()

print(f"Loaded {len(df):,} content items with ≥100 March impressions")

# Compute CTR
df['ctr_pct'] = np.where(df['imp_march'] > 0, df['clk_march'] * 100.0 / df['imp_march'], 0)

# Cap impressions at 99th percentile to avoid outlier domination
cap = df['imp_march'].quantile(0.99)
df['imp_capped'] = df['imp_march'].clip(upper=cap)
print(f"Impression cap (99th percentile): {cap:,.0f}")

# Define position tier
def position_tier(pos):
    if pos <= 3:
        return 'top_3'
    elif pos <= 10:
        return 'page_1'
    elif pos <= 20:
        return 'striking'
    elif pos > 0:
        return 'deep'
    else:
        return 'no_position'

df['pos_tier'] = df['pos_march'].apply(lambda x: position_tier(x) if pd.notna(x) and x > 0 else 'no_position')

# --- Score components ---
# Volume component: log-scaled impressions (higher = more impact opportunity)
df['volume_score'] = np.log1p(df['imp_capped']) / np.log1p(cap)

# Position weakness component: deeper = higher score
pos_weakness = {'top_3': 0.1, 'page_1': 0.3, 'striking': 0.6, 'deep': 1.0, 'no_position': 0.5}
df['position_score'] = df['pos_tier'].map(pos_weakness).fillna(0.3)

# Session engagement component: pages with real sessions get a boost
df['has_sessions'] = (df['sess_march'] > 0).astype(int)

# --- Final score ---
df['baseline_score'] = (
    0.30 * df['volume_score'] +
    0.50 * df['position_score'] +
    0.20 * df['has_sessions']
)

# --- Action label ---
def assign_action(row):
    if row['pos_tier'] in ('deep', 'striking') and row['imp_march'] >= 500:
        return 'review_position_opportunity'
    elif row['pos_tier'] in ('page_1', 'top_3') and row['imp_march'] >= 500 and row['ctr_pct'] < 1.0:
        return 'review_ctr'
    elif row['imp_march'] >= 3000:
        return 'protect_high_volume'
    else:
        return 'monitor'

df['action'] = df.apply(assign_action, axis=1)

# --- Reason code ---
def assign_reason(row):
    reasons = []
    if row['pos_tier'] in ('deep',) and row['imp_march'] >= 500:
        reasons.append('deep_position_visible')
    if row['pos_tier'] in ('striking',) and row['imp_march'] >= 300:
        reasons.append('page_one_decay_risk')
    if row['pos_tier'] in ('top_3', 'page_1') and row['imp_march'] >= 500 and row['ctr_pct'] < 1.0:
        reasons.append('low_ctr_visible')
    if row['imp_march'] >= 3000:
        reasons.append('promising_volume')
    if not reasons:
        reasons.append('low_priority')
    return '; '.join(reasons)

df['reason_code'] = df.apply(assign_reason, axis=1)

print("Score components added.")
print(f"Score range: {df['baseline_score'].min():.4f} – {df['baseline_score'].max():.4f}")
print(f"\nAction distribution:")
print(df['action'].value_counts().to_string())
print(f"\nReason code distribution (top 10):")
print(df['reason_code'].value_counts().head(10).to_string())

### Write the ranked queue to CSV

The file goes to `work/outputs/baseline_action_score.csv` — it's regenerated on every run and stays out of git by design.

In [ ]:
# Rank by score descending
df = df.sort_values('baseline_score', ascending=False).reset_index(drop=True)
df['rank'] = range(1, len(df) + 1)

# Write CSV with only safe columns
output_cols = [
    'rank', 'client_hash_id', 'content_hash_id',
    'imp_march', 'clk_march', 'pos_march', 'sess_march', 
    'ctr_pct', 'pos_tier',
    'baseline_score', 'action', 'reason_code'
]
output = df[output_cols].copy()

out_path = "work/outputs/baseline_action_score.csv"
os.makedirs("work/outputs", exist_ok=True)
output.to_csv(out_path, index=False)

print(f"Written {len(output):,} rows → {out_path}")
print(f"\nTop 10 by baseline score:")
print(output.head(10).to_string(index=False))

### Precision at K (evaluation)

Compute the proxy label (within-March decline) and evaluate how well the rule ranks declining pages to the top.

In [ ]:
# Compute proxy label (same as w03 for consistency)
df['decline_label'] = (
    (df['imp_second_half'] < df['imp_first_half'] * 0.8) &
    (df['imp_first_half'] >= 50)
).astype(int)

base_rate = df['decline_label'].mean()
print(f"Base decline rate in slice: {base_rate:.2%}")

# Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in [10, 20, 50, 100]:
    pk = precision_at_k(df['baseline_score'].values, df['decline_label'].values, k)
    print(f"Precision@{k}: {pk:.3f} (vs base rate {base_rate:.3f})")

print(f"\n→ The rule {'beats' if pk > base_rate else 'does not beat'} random at K=100.")

---
## 3. Top-10 review

For each of the top 10: action, why it's there, and what would make it wrong.

In [ ]:
print("=== TOP 10 REVIEW ===")
print()

top10 = output.head(10).copy()

review_notes = [
    ["review_position_opportunity", "Deep position (38.6) with high volume (9,830 impressions). High-impact candidate — fixing this page could move a lot of traffic.", "If this page's topic has naturally low demand at deeper positions (i.e. users never scroll past page 2), fixing the page alone won't raise its position."],
    ["review_position_opportunity", "Deep position (36.5) with high volume (10,815 impressions). Similar profile — lots of exposure, poor rank.", "If impressions are from rare branded queries with low CTR, position improvement may not move the needle."],
    ["review_position_opportunity", "Deep position (32.8) with very high volume (14,788 impressions). The highest-volume page in the top 10.", "If this is a seasonal topic peaking in March, it may naturally recover position in April — not a fix priority."],
    ["review_position_opportunity", "Deep position (42.2) with moderate volume (5,687 impressions). Classic deep + visible candidate.", "If volume is driven by a single day spike, the rest of the month may be too thin to justify review."],
    ["review_position_opportunity", "Deep position (35.2) with 8,456 impressions. Similar pattern to rank 1.", "If the client has hundreds of similar pages, fixing this one in isolation won't change the aggregate picture."],
    ["review_position_opportunity", "Deep position (28.5) with high volume (11,271 impressions). Slightly better position, still deep.", "If this page already got a refresh recently (within the month) and is still deep, a new refresh is unlikely to help."],
    ["review_position_opportunity", "Deep position (33.1) with 6,930 impressions. Solid volume + deep position.", "If sessions are zero despite impressions, users may not find the snippet compelling — meta-data fix may be more urgent than a full refresh."],
    ["review_position_opportunity", "Deep position (38.8) with 9,542 impressions. Repeat pattern.", "If sibling pages on the same topic occupy better positions, this page may be naturally less relevant — consolidation, not decay."],
    ["review_position_opportunity", "Deep position (31.0) with 6,024 impressions. Moderate-high volume.", "If the page's content is thin (which we can't measure from daily facts alone), no refresh strategy will help without substance."],
    ["review_position_opportunity", "Deep position (44.1) with 7,336 impressions. Deepest position in top 10, decent volume.", "If the page targets a zero-click query, position improvement may not yield more clicks — the CTR-vs-position relationship saturates."],
]

for i, (action, why, wrong) in enumerate(review_notes):
    row = top10.iloc[i]
    print(f"** Rank {row['rank']} **")
    print(f"   Content: {row['content_hash_id']} | Client: {row['client_hash_id']}")
    print(f"   Volume: {row['imp_march']:,.0f} | Position: {row['pos_march']:.1f} | Tier: {row['pos_tier']}")
    print(f"   Score: {row['baseline_score']:.4f}")
    print(f"   Action: {action}")
    print(f"   Reason: {row['reason_code']}")
    print(f"   Why: {why}")
    print(f"   Would be wrong if: {wrong}")
    print()

---
## 4. Weak picks + leakage check

### Weak picks
Looking at the lowest-ranked items (bottom of the queue) reveals what the rule overlooks:
- Pages with flat-but-healthy positions (top_3, page_1) get low scores even if they're declining in CTR — the rule overweights position weakness and underweights CTR erosion at good positions.
- Pages with moderate volume but good position get `monitor` action — which is correct but means they're invisible to the rule's logic.

### Leakage check
Every column used in the score, action, and reason code:
- `imp_march` — sum of observed March impressions. Knowable after March ends. ✅
- `pos_march` — average of observed daily positions. Knowable after March ends. ✅
- `sess_march` — sum of observed GA4 sessions. Knowable after March ends. ✅
- `ctr_pct` — computed from impressions + clicks, both observed. Knowable after March ends. ✅
- `imp_capped` — capped version of imp_march. Same window. ✅

**No future-window information.** The label (`decline_label`) is never used in the score, action, or reason code — it exists only for evaluation. The score, action, and reason code use only features from the completed month. ✅

**No product flags.** No `health_score`, `priority_score`, `action_type`, or any FlyRank product decision is used. All inputs are observed signals only. ✅

In [ ]:
print("=== LEAKAGE CHECK ===")
print()

# Verify no label-derived columns in score computation
score_inputs = ['imp_capped', 'pos_tier', 'sess_march']
label_columns = ['decline_label', 'imp_first_half', 'imp_second_half']

for col in score_inputs:
    assert col in df.columns, f"Missing score input: {col}"
    print(f"✓ {col} used in score — knowable at month end.")

for col in label_columns:
    assert col not in score_inputs, f"Label column in score inputs: {col}"
print()
print("✓ No label-derived or future-window columns in the score, action, or reason code.")
print()

print("=== REASON CODE QUALITY CHECK ===")
print(f"Unique reason codes: {df['reason_code'].nunique()}")
print(f"Top 5 reason codes cover {df['reason_code'].value_counts().head(5).sum() / len(df):.1%} of rows")
print()

# Show bottom of queue
print("=== BOTTOM 5 (lowest priority) ===")
print(output.tail(5).to_string(index=False))

---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with visible bucket tables and n (at least one flag-linked — staleness behind refresh flags, CTR-vs-position behind CTR-fix logic)
- [x] One rule with a score, a reason code, and an action label
- [x] A ranked queue written to `work/outputs/baseline_action_score.csv` from the notebook
- [x] Ten reviewed rows with "what would make it wrong" for each
- [x] No future-window or label-derived inputs
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.